In [52]:
import multiprocessing
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

from sklearn.preprocessing import FunctionTransformer

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import torch
from sentence_transformers import SentenceTransformer

# Load a pre-trained SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Move the model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print('GPU : ', torch.cuda.get_device_name(0))

GPU :  NVIDIA GeForce RTX 3070


In [53]:
# Load datasets
# Lecture des jeux de données
train = pd.read_excel('../data/training_datasets/train_dataset_40pc.xlsx')
test = pd.read_excel('../data/test_dataset_10.xlsx')

train['text_post'] = train['text_post']
test['text_post'] = test['text_post']

train['category'] = train['category'].apply(lambda x: 1 if x == 'incel' else 0)
test['category'] = test['category'].apply(lambda x: 1 if x == 'incel' else 0)

X_train, y_train = train.text_post, train.category
X_test, y_test = test.text_post, test.category

In [54]:
sbert_embedder = FunctionTransformer(
    lambda x : model.encode(
        x.astype(str).values,
        batch_size=32, 
        convert_to_numpy=True, 
        show_progress_bar=True,
        device=device)
)

In [58]:
# Définition du pipeline

pipeline = Pipeline(
    [
        ("vectorizer", sbert_embedder),
        ("classify", "passthrough"),
    ]
)

param_grid = [
    {
        "classify" : [
            LogisticRegression(n_jobs=1), 
            LinearSVC(dual="auto"),
            GaussianNB(),
            KNeighborsClassifier(n_neighbors=5, n_jobs=1),
            RandomForestClassifier(n_jobs=1)
            ]
    }
]

grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    n_jobs = multiprocessing.cpu_count()-1, 
    verbose=1, 
    refit='f1_macro', 
    scoring=['accuracy','f1_macro']
)

In [59]:
grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    n_jobs = multiprocessing.cpu_count()-1, 
    verbose=1, 
    refit='f1_macro', 
    scoring=['accuracy','f1_macro']
)

In [ ]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


In [ ]:
grid_search.best_params_

{'classify': LinearSVC()}

In [ ]:
results_cv = pd.DataFrame(grid_search.cv_results_)
results_cv = results_cv[
    ['param_classify', #'param_vectorizer__max_features', 
     'split0_test_accuracy', 'split1_test_accuracy', 'split2_test_accuracy',
       'split3_test_accuracy', 'split4_test_accuracy', 'mean_test_accuracy',
       'std_test_accuracy', 'rank_test_accuracy', 'split0_test_f1_macro',
       'split1_test_f1_macro', 'split2_test_f1_macro', 'split3_test_f1_macro',
       'split4_test_f1_macro', 'mean_test_f1_macro', 'std_test_f1_macro',
       'rank_test_f1_macro']
    ]

results_cv.sort_values(by='rank_test_f1_macro')

,param_classify,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro
1,LinearSVC(),0.610875,0.684250,0.859875,0.849750,0.835125,0.767975,0.101323,1,0.379219,0.406264,0.462329,0.459386,0.455078,0.432455,0.033614,1
0,LogisticRegression(n_jobs=3),0.610125,0.686125,0.859375,0.847375,0.831750,0.766950,0.100336,2,0.378930,0.406924,0.462185,0.458691,0.454074,0.432161,0.033334,2
3,RandomForestClassifier(n_jobs=3),0.396375,0.542875,0.896875,0.873500,0.864875,0.714900,0.205821,3,0.283860,0.351859,0.472817,0.466240,0.463771,0.407709,0.076506,3
2,MultinomialNB(),NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
